# UCI Adult Income Dataset Bias Analysis
## Analyzing and Mitigating Gender Bias in Income Predictions

## 1. Setup and Installation

In [ ]:
# Install required packages
!pip install pandas numpy matplotlib aif360 scikit-learn --quiet

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from aif360.datasets import BinaryLabelDataset
from aif360.metrics import BinaryLabelDatasetMetric
from aif360.algorithms.preprocessing import Reweighing

%matplotlib inline
print("✅ Packages loaded successfully")

## 2. Data Loading and Preparation

In [ ]:
# Load dataset from UCI repository
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"
cols = ['age', 'workclass', 'education', 'gender', 'race', 'income']

df = pd.read_csv(url, names=cols, na_values=' ?', skipinitialspace=True).dropna()

# Convert to binary categories
df['income'] = (df['income'] == '>50K').astype(int)
df['gender'] = (df['gender'] == 'Male').astype(int)
df['race'] = (df['race'] == 'White').astype(int)

print(f"✅ Data loaded successfully! {len(df)} records.")
print("\nFirst 3 rows:")
df.head(3)

## 3. Initialize Fairness Analysis

In [ ]:
# Create AIF360 dataset
dataset = BinaryLabelDataset(
    df=df,
    label_names=['income'],
    favorable_classes=[1],  # >50K is favorable
    protected_attribute_names=['gender'],
    privileged_classes=[[1]]  # Male=privileged
)

# Split data (80% train, 20% test)
train, test = dataset.split([0.8], shuffle=True, seed=42)
print("✅ Dataset prepared with train/test split")

## 4. Calculate Initial Bias Metrics

In [ ]:
# Initialize metrics
metric = BinaryLabelDatasetMetric(
    test,
    unprivileged_groups=[{'gender': 0}],  # Female=unprivileged
    privileged_groups=[{'gender': 1}]
)

print("\n📊 Initial Fairness Metrics:")
print(f"Statistical Parity Difference: {metric.statistical_parity_difference():.3f} (Closer to 0 is fairer)")
print(f"Disparate Impact Ratio: {metric.disparate_impact():.3f} (Closer to 1 is fairer)")

# Show distribution
print("\n📈 Income distribution by gender:")
pd.crosstab(df['gender'], df['income'], normalize='index')

## 5. Apply Bias Mitigation (Reweighing)

In [ ]:
# Apply reweighing
RW = Reweighing(
    unprivileged_groups=[{'gender': 0}],
    privileged_groups=[{'gender': 1}]
)
dataset_transformed = RW.fit_transform(dataset)

# Recalculate metrics
train_transformed, test_transformed = dataset_transformed.split([0.8], seed=42)
metric_transformed = BinaryLabelDatasetMetric(
    test_transformed,
    unprivileged_groups=[{'gender': 0}],
    privileged_groups=[{'gender': 1}]
)

print("\n⚖️ Post-Mitigation Metrics:")
print(f"Statistical Parity Difference: {metric_transformed.statistical_parity_difference():.3f}")
print(f"Disparate Impact Ratio: {metric_transformed.disparate_impact():.3f}")

## 6. Visualize Results

In [ ]:
# Prepare data
metrics = ['Statistical Parity', 'Disparate Impact']
before = [metric.statistical_parity_difference(), metric.disparate_impact()]
after = [metric_transformed.statistical_parity_difference(), metric_transformed.disparate_impact()]

# Create visualization
plt.figure(figsize=(10, 5))
x = np.arange(len(metrics))
width = 0.35

plt.bar(x - width/2, before, width, label='Before', color='#1f77b4')
plt.bar(x + width/2, after, width, label='After', color='#ff7f0e')

# Add reference lines
plt.axhline(y=0, color='black', linestyle='--', alpha=0.5)
plt.axhline(y=1, color='green', linestyle='--', alpha=0.5)

# Formatting
plt.title('Fairness Metrics Before/After Mitigation')
plt.ylabel('Metric Value')
plt.xticks(x, metrics)
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Save Results

In [ ]:
# Save processed data
df.to_csv('adult_income_processed.csv', index=False)

# Save visualization
plt.savefig('fairness_comparison.png', dpi=300, bbox_inches='tight')

print("💾 Results saved to:")
print("- adult_income_processed.csv")
print("- fairness_comparison.png")

## Key Findings
- Initial analysis shows gender bias in income predictions
- Reweighing technique improved fairness metrics
- Further improvements could be explored with:
  - Different mitigation algorithms
  - Feature engineering
  - Model selection